# 2 · Measure accuracy on IMAGES and then VIDEOS (one after the other)

Run the cells **top to bottom**. Use a GPU: *Runtime → Change runtime type → T4 GPU*.

You need two things: **(1)** the project zip (`deepfake_detection_suite_FINAL.zip`) and **(2)** a labelled-data folder (or zip) laid out like this:

```
UPDATE/
  REAL IMAGE/   FAKE IMAGE/     (jpg, png, webp...)
  REAL VIDEO/   FAKE VIDEO/     (mp4, avi, mov...)
```
(`images/real`, `videos/fake` etc. work too.)
Round 1 uses small samples so you can confirm everything works in a few minutes. Then raise `N_IMG` / `N_VID` and re-run from section 4.

## 1. Install

In [ ]:
!pip install -q torch transformers pillow numpy opencv-python-headless pymupdf librosa soundfile scikit-image pandas
print("Installed.")

## 2. Load the project code (upload the project zip when asked)

In [ ]:
import os, glob, zipfile
from google.colab import files
up = files.upload()                       # choose deepfake_detection_suite_FINAL.zip
with zipfile.ZipFile(list(up.keys())[0]) as z:
    z.extractall("/content/project")
hits = glob.glob("/content/project/**/06_dataset_and_evaluation/batch_evaluate.py", recursive=True)
assert hits, "batch_evaluate.py not found - did you upload the FINAL project zip?"
SUITE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print("Suite root:", SUITE_ROOT)

## 2b. (Optional) Test with your fine-tuned image model

Set `USE_FINETUNED = True` to evaluate with the model from notebook 3 (images AND video frames use it). Leave it `False` to evaluate the original pretrained model.

In [ ]:
USE_FINETUNED = False         # True = use the model trained in notebook 3
FROM_DRIVE = True             # True = read MyDrive/image_finetuned.zip, False = upload it

if USE_FINETUNED:
    import shutil
    if FROM_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zpath = "/content/drive/MyDrive/image_finetuned.zip"
    else:
        up = files.upload()
        zpath = list(up.keys())[0]
    models_dir = os.path.join(SUITE_ROOT, "models")
    os.makedirs(models_dir, exist_ok=True)
    shutil.rmtree(os.path.join(models_dir, "image_finetuned"), ignore_errors=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(models_dir)
    assert os.path.isfile(os.path.join(models_dir, "image_finetuned", "config.json")), "zip must contain an image_finetuned/ folder"
    print("Fine-tuned model installed:", os.path.join(models_dir, "image_finetuned"))
else:
    print("Using the original pretrained model.")


## 3. Load your labelled data
Folder names can be anything that contains **real** / **fake** and **image** / **video**, e.g. `REAL IMAGE`, `FAKE VIDEO`, or `images/real`.
**Option A (recommended for big video files):** put the folder in Google Drive and set `USE_DRIVE = True`.
**Option B:** leave `USE_DRIVE = False` and upload a zip of the folder when asked (slow for large videos).

In [ ]:
USE_DRIVE = False
DRIVE_FOLDER = "/content/drive/MyDrive/UPDATE"     # only used when USE_DRIVE = True (the folder that holds your REAL/FAKE folders)

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
VID_EXT = (".mp4", ".avi", ".mov", ".mkv", ".webm")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = DRIVE_FOLDER
else:
    up = files.upload()
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("/content/labelled")
    SRC = "/content/labelled"

def discover(base):
    """Finds labelled files from folder names. Understands 'images/real', 'REAL IMAGE', 'fake_videos', 'Real Video'...
    label = a folder name containing 'real' or 'fake'; type = a folder name containing 'image'/'video' (else the file extension)."""
    data, skipped_ext, unlabelled = {}, {}, 0
    for root, dirs, fs in os.walk(base):
        dirs[:] = [d for d in dirs if d != "__MACOSX"]
        rel = os.path.relpath(root, base).lower().replace("\\", "/")
        parts = [] if rel == "." else rel.split("/")
        label = next(("fake" if "fake" in p else "real" for p in reversed(parts) if "fake" in p or "real" in p), None)
        mod = next(("videos" if "video" in p else "images" for p in reversed(parts)
                    if "video" in p or "image" in p or "img" in p or "photo" in p), None)
        for f in fs:
            if f.startswith("."):
                continue
            ext = os.path.splitext(f)[1].lower()
            m = mod or ("videos" if ext in VID_EXT else "images" if ext in IMG_EXT else None)
            if m is None or label is None:
                unlabelled += 1
                continue
            if ext not in (VID_EXT if m == "videos" else IMG_EXT):
                skipped_ext[ext or "(no extension)"] = skipped_ext.get(ext or "(no extension)", 0) + 1
                continue
            data.setdefault((m, label), []).append(os.path.join(root, f))
    return {k: sorted(v) for k, v in data.items()}, skipped_ext, unlabelled

DATA, skipped_ext, unlabelled = discover(SRC)
print("Data root:", SRC)
for mod in ("images", "videos"):
    print(f"  {mod:6s}  real={len(DATA.get((mod,'real'),[])):5d}   fake={len(DATA.get((mod,'fake'),[])):5d}")
if skipped_ext:
    print("  ignored (unsupported type):", skipped_ext)
if unlabelled:
    print(f"  ignored {unlabelled} files whose folder name has no 'real'/'fake'")
assert DATA, "No labelled files found. Folder names must contain 'real' or 'fake' (e.g. 'REAL VIDEO', 'images/fake')."

## 4. Choose how many files to test (per class)
Round 1 (trial): keep the defaults. Round 2: images 300+, and for videos `None` = use all of them. The sample is random but reproducible.

In [ ]:
import random, shutil
N_IMG = 100        # images per class (None = all)
N_VID = 10         # videos per class (None = all)
EVAL = "/content/eval_data"

shutil.rmtree(EVAL, ignore_errors=True)
for mod, n in (("images", N_IMG), ("videos", N_VID)):
    for label in ("real", "fake"):
        fl = DATA.get((mod, label), [])
        if n and len(fl) > n:
            fl = sorted(random.Random(0).sample(fl, n))
        if not fl:
            continue
        out = os.path.join(EVAL, mod, label)
        os.makedirs(out, exist_ok=True)
        for i, f in enumerate(fl):
            stem, ext = os.path.splitext(os.path.basename(f))
            name = f"{i:05d}_{stem}" + (".mp4" if mod == "videos" else ext.lower())   # evaluator reads *.mp4
            os.symlink(f, os.path.join(out, name))
        print(f"{mod:6s} {label}: {len(fl)} files")

## 5. IMAGES: run the evaluation
Runs the image model four ways (model only / + face crop / + forensic fusion / both), sweeps the UNCERTAIN band,
and fits + checks a calibration. Progress lines appear as each variant starts.

In [ ]:
import subprocess, sys
os.makedirs("/content/eval_out", exist_ok=True)

def run(args):
    cmd = [sys.executable, "-u", f"{SUITE_ROOT}/06_dataset_and_evaluation/batch_evaluate.py", *args]
    p = subprocess.Popen(cmd, cwd="/content/eval_out", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("\n[exit code]", p.returncode)

run(["--data", EVAL, "--modalities", "images", "--compare-variants", "--fit-calibration",
     "--calibration_dir", "/content/eval_out/calibration",
     "--save_csv", "/content/eval_out/images_results.csv", "--save_json", "/content/eval_out/images_report.json"])

## 6. VIDEOS: run the evaluation
Each video is slow (frames + audio track), and **nothing prints until it finishes** - roughly 10-30 seconds per video on a T4.
With `N_VID = 10` per class expect about 5 minutes. The evaluator samples 6 frames per video.

In [ ]:
run(["--data", EVAL, "--modalities", "videos",
     "--save_csv", "/content/eval_out/videos_results.csv", "--save_json", "/content/eval_out/videos_report.json"])

## 7. How to read the numbers
* **ROC-AUC**: 0.5 = cannot tell real from fake on your data; 0.9+ = separates well. The most honest single number.
* **accuracy (strict)** counts UNCERTAIN as wrong; **decided-only** counts just the answered files; read it with **coverage**.
* **band sweep**: each row is one UNCERTAIN band - pick a row you can defend.
* **best cut-off**: far from 50 means the model is biased on your data.
* **variant table** (images): keep face crop / fusion only if they beat "model only".
* **calibration holdout**: ECE (points) before/after on files not used for fitting; lower is better.
* With only ~20 files per class every number is noise - treat Round 1 as a plumbing check.
* **Calibration file** (`eval_out/calibration/calibration_image.json`, images only, optional): keep it ONLY if the model already separates your data well (AUC about 0.9 or higher). One fitted on a near-chance model just squashes every score toward 50% - do not install that one.
* **Test on images from a DIFFERENT source** than the data a model was trained on (other cameras, other AI generators). Accuracy on the training source does not transfer automatically.

## 8. The mistakes worth looking at, and download everything

In [ ]:
import pandas as pd
for name in ("images", "videos"):
    p = f"/content/eval_out/{name}_results.csv"
    if not os.path.exists(p):
        print(name, ": no results file"); continue
    df = pd.read_csv(p)
    print("=" * 60, "\n", name.upper(), f"({len(df)} files)")
    print(pd.crosstab(df.ground_truth_label, df.predicted_verdict))
    is_fake = df.ground_truth_label.eq("fake")
    df["wrong_by"] = (df.confidence_fake_pct - 50).where(~is_fake, 50 - df.confidence_fake_pct)
    wrong = df[df.wrong_by > 0]
    print(f"\nWrong answers (incl. UNCERTAIN leaning the wrong way): {len(wrong)}. Most confident first:")
    print(wrong.sort_values("wrong_by", ascending=False).head(8)[["file", "ground_truth_label", "predicted_verdict", "confidence_fake_pct"]].to_string(index=False))

In [ ]:
shutil.make_archive("/content/eval_results", "zip", "/content/eval_out")
files.download("/content/eval_results.zip")